In [ ]:
import pandas as pd 
df_single = pd.read_csv("52_매출액_개별.csv",encoding="cp949").set_index(['업체코드'])
df_link = pd.read_csv("52_매출액_연결.csv",encoding="cp949").set_index(['업체코드'])

In [3]:
import pandas as pd 
df_single = pd.read_csv("53_유동자산_개별.csv",encoding="cp949").set_index(['업체코드'])
df_link = pd.read_csv("53_유동자산_연결.csv",encoding="cp949").set_index(['업체코드'])

# 3. 연도별 컬럼 리스트 생성 (_2014 ~ _2026)
years = [f"_{yr}" for yr in range(2014, 2027)]

# 4. 통합 작업 (핵심 로직)
# df_link의 복사본을 만들어서 작업을 시작합니다.
df_final = df_link.copy()

for yr in years:
    # 각 연도별로 연결(link)이 Null인 곳만 개별(single) 값으로 채움
    df_final[yr] = df_final[yr].fillna(df_single[yr])

# 5. 인덱스 해제 (업체코드를 다시 일반 컬럼으로 돌림)
df_final = df_final.reset_index()

# 6. 결과 저장 (한글 깨짐 방지 포함)
df_final.to_csv("53_유동자산_통합.csv", index=False, encoding="cp949")
print("✅ 연결/개별 통합 및 파일 저장 완료!")

✅ 연결/개별 통합 및 파일 저장 완료!


In [ ]:
import pandas as pd
import glob
import os

# 1. 처리할 번호 리스트 생성 (1~24, 32~44, 52)
target_numbers = list(range(1, 25)) + list(range(32, 45)) + [52]
# 숫자를 '01', '02' 처럼 두 자리 문자열로 변환
target_prefixes = [f"{num:02d}" for num in target_numbers]

# 2. 연도 리스트 (_2014 ~ _2026)
years = [f"_{yr}" for yr in range(2014, 2027)]

for prefix in target_prefixes:
    # 해당 번호로 시작하는 파일들 찾기
    files = glob.glob(f"{prefix}_*.csv")
    
    # 짝꿍 파일 찾기 (개별 vs 연결)
    file_single = ""
    file_link = ""
    
    for f in files:
        if "(개별)" in f:
            file_single = f
        elif "통합" not in f: # 이미 만든 통합 파일은 제외
            file_link = f
            
    # 파일이 둘 다 존재할 때만 실행
    if file_single and file_link:
        print(f"🔄 작업 중: {prefix}번 ({file_link} + {file_single})")
        
        try:
            # 데이터 불러오기 (Unnamed 컬럼 방어 포함)
            df_s = pd.read_csv(file_single, encoding="cp949").set_index('업체코드')
            df_l = pd.read_csv(file_link, encoding="cp949").set_index('업체코드')
            
            # 불필요한 Unnamed 컬럼 제거
            df_s = df_s.loc[:, ~df_s.columns.str.contains('^Unnamed')]
            df_l = df_l.loc[:, ~df_l.columns.str.contains('^Unnamed')]
            
            # 보완 작업 (combine_first가 더 깔끔하므로 권장)
            df_final = df_l.combine_first(df_s)
            
            # 파일명 생성 (예: 01_총자본증가율_통합.csv)
            # 원본 파일명에서 '(개별)' 등을 떼고 이름 추출
            base_name = file_link.replace(".csv", "")
            save_name = f"{base_name}_통합.csv"
            
            # 저장
            df_final.reset_index().to_csv(save_name, index=False, encoding="cp949")
            print(f"✅ 저장 완료: {save_name}")
            
        except Exception as e:
            print(f"❌ {prefix}번 처리 중 오류 발생: {e}")
    else:
        print(f"⚠️ {prefix}번 파일을 찾을 수 없어 건너뜁니다.")

print("\n🚀 모든 지정된 번호의 통합 작업이 완료되었습니다!")